# 지면 YOLO-seg TFLite 변환_Local

Colab에서 학습한 `best.pt`를 로컬 `Downloads`에 내려받은 뒤 TFLite integer/full integer 모델로 변환합니다.

입력:

```text
Downloads/best.pt
C:/Dataset/Train2YOLO_Outdoor_SurfaceGuideSeg/data.yaml
```

출력:

```text
Downloads/OutdoorSurfaceGuide_TFLite/outdoor_surface_guide_yolo26n-seg_integer_quant.tflite
```


In [1]:
# 라즈베리파이 TensorFlow Lite 2.15.0 런타임과 맞추기 위한 변환 환경입니다.
# 이 셀을 실행한 뒤 반드시 커널을 재시작하고, 아래 셀들을 처음부터 다시 실행하세요.
# 최신 onnx는 TensorFlow 2.15가 요구하는 낮은 ml-dtypes와 충돌할 수 있어 함께 고정합니다.
# %pip uninstall -y tensorflow tensorflow-cpu keras tf-keras ml-dtypes onnx onnxslim onnxruntime onnx2tf
%pip install "numpy<2" "protobuf<5" "ml-dtypes==0.2.0" "tensorflow==2.15.0" "keras==2.15.0" "tf-keras==2.15.0" "onnx==1.16.1" "onnxslim==0.1.34" ultralytics pyyaml


  Using cached protobuf-4.25.9-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached onnxslim-0.1.34-py3-none-any.whl.metadata (2.7 kB)
Using cached onnxslim-0.1.34-py3-none-any.whl (140 kB)
Using cached protobuf-4.25.9-cp310-abi3-win_amd64.whl (413 kB)

  Attempting uninstall: protobuf

    Found existing installation: protobuf 7.34.1

    Uninstalling protobuf-7.34.1:

      Successfully uninstalled protobuf-7.34.1

   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
  Attempting uninstall: onnxslim
   ---------------------------------------- 0/2 [protobuf]
    Found existing installation: onnxslim 0.1.93
   ---------------------------------------- 0/2 [protobuf]
    Uninstalling onnxslim-0.1.93:
   ---------------------------------------- 0/2 [protobuf]
      Successfully uninstalled onnxslim-0.1.93
   ---------------------------------------- 0/2 [proto

In [2]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

try:
    import tensorflow as tf
    print("tensorflow:", tf.__version__)
except Exception as e:
    raise RuntimeError(f"TensorFlow를 불러오지 못했습니다: {e}")

# Pi의 tflite_runtime/tensorflow-lite 런타임이 2.15.0이면 2.19.x에서 변환한 모델이
# TRANSPOSE_CONV 양자화 호환성 문제를 일으킬 수 있습니다.
if not tf.__version__.startswith("2.15."):
    raise RuntimeError(
        "현재 TensorFlow 버전이 2.15.x가 아닙니다. "
        "라즈베리파이 2.15.0 런타임용 TFLite는 TensorFlow 2.15.x 환경에서 다시 변환하세요."
    )

for package in ["ai_edge_litert", "tflite_runtime"]:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "installed")
        print(f"{package}: {version}")
    except Exception as e:
        print(f"{package}: not available ({e})")


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Executable: c:\Users\KCCISTC\AppData\Local\Programs\Python\Python311\python.exe

tensorflow: 2.15.0
ai_edge_litert: 2.1.4
tflite_runtime: not available (No module named 'tflite_runtime')


In [3]:
from pathlib import Path
import os
import random
import shutil
import yaml

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

DOWNLOADS = Path.home() / 'Downloads'
MODEL_PT = DOWNLOADS / 'best.pt'
SOURCE_ROOT = Path('C:/Dataset/Train2YOLO_Outdoor_SurfaceGuideSeg')
SOURCE_DATA_YAML = SOURCE_ROOT / 'data.yaml'
EXPORT_ROOT = DOWNLOADS / 'OutdoorSurfaceGuide_TFLite'
CALIB_ROOT = EXPORT_ROOT / 'calib_dataset'
CALIB_DATA_YAML = CALIB_ROOT / 'data.yaml'

IMGSZ = 512
CALIB_IMAGES_PER_SPLIT = 500
CALIB_RANDOM_SEED = 42

if not MODEL_PT.exists():
    raise FileNotFoundError(MODEL_PT)
if not SOURCE_DATA_YAML.exists():
    raise FileNotFoundError(SOURCE_DATA_YAML)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

print('MODEL_PT:', MODEL_PT)
print('SOURCE_DATA_YAML:', SOURCE_DATA_YAML)
print('EXPORT_ROOT:', EXPORT_ROOT)


MODEL_PT: C:\Users\KCCISTC\Downloads\best.pt
SOURCE_DATA_YAML: C:\Dataset\Train2YOLO_Outdoor_SurfaceGuideSeg\data.yaml
EXPORT_ROOT: C:\Users\KCCISTC\Downloads\OutdoorSurfaceGuide_TFLite


## calibration 데이터셋 생성

In [4]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
random.seed(CALIB_RANDOM_SEED)

source_data = yaml.safe_load(SOURCE_DATA_YAML.read_text(encoding='utf-8'))
names = source_data['names']

for split in ['train', 'val']:
    src_img_dir = SOURCE_ROOT / 'images' / split
    src_lbl_dir = SOURCE_ROOT / 'labels' / split
    dst_img_dir = CALIB_ROOT / 'images' / split
    dst_lbl_dir = CALIB_ROOT / 'labels' / split
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = sorted(p for p in src_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    selected = images if len(images) <= CALIB_IMAGES_PER_SPLIT else random.sample(images, CALIB_IMAGES_PER_SPLIT)
    for img in selected:
        shutil.copy2(img, dst_img_dir / img.name)
        src_lbl = src_lbl_dir / f'{img.stem}.txt'
        dst_lbl = dst_lbl_dir / f'{img.stem}.txt'
        if src_lbl.exists():
            shutil.copy2(src_lbl, dst_lbl)
        else:
            dst_lbl.touch(exist_ok=True)
    print(split, len(selected))

calib_data = {'path': str(CALIB_ROOT), 'train': 'images/train', 'val': 'images/val', 'names': names}
CALIB_DATA_YAML.write_text(yaml.safe_dump(calib_data, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(CALIB_DATA_YAML)


train 500
val 500
C:\Users\KCCISTC\Downloads\OutdoorSurfaceGuide_TFLite\calib_dataset\data.yaml


## TFLite 변환

In [5]:
from ultralytics import YOLO

# export 직전에도 한 번 더 검사합니다. 이 셀만 단독 실행해도 2.19.x 변환을 막기 위함입니다.
import tensorflow as tf
if not tf.__version__.startswith('2.15.'):
    raise RuntimeError(
        f'현재 TensorFlow는 {tf.__version__}입니다. '
        '라즈베리파이 2.15.0 런타임용 변환은 TensorFlow 2.15.x에서만 실행하세요. '
        '설치 셀 실행 후 커널을 재시작해야 합니다.'
    )
print('변환 TensorFlow:', tf.__version__)



MODEL_STEM = 'outdoor_surface_guide_yolo26n-seg'
FINAL_TFLITE_NAME = 'outdoor_surface_guide_yolo26n-seg_integer_quant.tflite'
FINAL_PT_NAME = 'outdoor_surface_guide_yolo26n-seg.pt'


def find_tflite_outputs(exported_path, roots):
    exported_path = Path(exported_path)
    search_roots = []
    if exported_path.is_file():
        search_roots.append(exported_path.parent)
    elif exported_path.is_dir():
        search_roots.append(exported_path)
    search_roots.extend(Path(r) for r in roots)

    found = []
    for root in search_roots:
        if root.exists():
            found.extend(root.rglob('*.tflite'))
    return sorted(set(found), key=lambda x: (x.stat().st_mtime, str(x)), reverse=True)


def pick_variant(candidates, suffix):
    matched = [
        p for p in candidates
        if p.name.endswith(suffix) and 'int16_act' not in p.name
    ]
    return matched[0] if matched else None


def backup_variant(candidates, suffix, target_name, required=False):
    src = pick_variant(candidates, suffix)
    if src is None:
        if required:
            print('TFLite 후보:')
            for p in candidates:
                print('-', p)
            raise FileNotFoundError(f'{suffix} 산출물을 찾지 못했습니다.')
        print(f'건너뜀: {suffix} 산출물이 없습니다.')
        return None

    dst = EXPORT_ROOT / target_name
    shutil.copy2(src, dst)
    print(f'백업: {src.name} -> {dst.name}')
    return dst


model = YOLO(str(MODEL_PT))

print('1) INT8/full integer 변환 시작')
exported_int8 = model.export(
    format='tflite',
    int8=True,
    data=str(CALIB_DATA_YAML),
    imgsz=IMGSZ,
    batch=1,
    device='cpu',
    nms=True,
)

int8_candidates = find_tflite_outputs(exported_int8, [MODEL_PT.parent, EXPORT_ROOT])
full_integer = backup_variant(
    int8_candidates,
    'full_integer_quant.tflite',
    f'{MODEL_STEM}_full_integer_quant.tflite',
    required=True,
)
backup_variant(int8_candidates, 'integer_quant.tflite', f'{MODEL_STEM}_integer_quant.tflite')
backup_variant(int8_candidates, 'int8.tflite', f'{MODEL_STEM}_int8.tflite')

# 기존 raspi5/config.py가 참조하던 파일명은 유지합니다.
final_tflite = EXPORT_ROOT / FINAL_TFLITE_NAME
shutil.copy2(full_integer, final_tflite)

print('\n2) float32 변환 시작')
exported_float32 = model.export(
    format='tflite',
    int8=False,
    imgsz=IMGSZ,
    batch=1,
    device='cpu',
    nms=True,
)

float32_candidates = find_tflite_outputs(exported_float32, [MODEL_PT.parent, EXPORT_ROOT])
backup_variant(float32_candidates, 'float32.tflite', f'{MODEL_STEM}_float32.tflite', required=True)

final_pt = EXPORT_ROOT / FINAL_PT_NAME
shutil.copy2(MODEL_PT, final_pt)

print('\n대표 TFLite:', final_tflite)
print('PT 백업:', final_pt)


변환 TensorFlow: 2.15.0
1) INT8/full integer 변환 시작
Ultralytics 8.4.50  Python-3.11.9 torch-2.12.0+cpu CPU (Intel Core i7-9700 3.00GHz)
WARNING 'nms=True' is not available for end2end models. Forcing 'nms=False'.
YOLO26n-seg summary (fused): 139 layers, 2,690,249 parameters, 0 gradients, 9.0 GFLOPs

PyTorch: starting from 'C:\Users\KCCISTC\Downloads\best.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) ((1, 300, 38), (1, 32, 128, 128)) (18.1 MB)
TensorFlow SavedModel: collecting INT8 calibration images from 'data=C:\Users\KCCISTC\Downloads\OutdoorSurfaceGuide_TFLite\calib_dataset\data.yaml'
Fast image access  (ping: 0.10.0 ms, read: 6.31.4 MB/s, size: 36.2 KB)
Scanning C:\Users\KCCISTC\Downloads\OutdoorSurfaceGuide_TFLite\calib_dataset\labels\val... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 786.9it/s 0.6s0.0s
C:\Users\KCCISTC\Downloads\OutdoorSurfaceGuide_TFLite\calib_dataset\images\val\outdoor_surface_Surface_1_Surface_030_MP_SEL_SUR_003347.jpg: 1 dupl

c:\Users\KCCISTC\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.93...
ONNX: export success  5.7s, saved as 'C:\Users\KCCISTC\Downloads\best.onnx' (10.6 MB)
requirements: Ultralytics requirement ['protobuf>=5'] not found, attempting AutoUpdate...
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.34.1 which is incompatible.

requirements: AutoUpdate success  9.4s
WARNING requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with te

## 결과 확인

In [6]:
print('EXPORT_ROOT:', EXPORT_ROOT)
for p in sorted(EXPORT_ROOT.glob('*.tflite')):
    print(p.name, round(p.stat().st_size / (1024 * 1024), 2), 'MB')


EXPORT_ROOT: C:\Users\KCCISTC\Downloads\OutdoorSurfaceGuide_TFLite
outdoor_surface_guide_yolo26n-seg_float32.tflite 10.71 MB
outdoor_surface_guide_yolo26n-seg_full_integer_quant.tflite 3.16 MB
outdoor_surface_guide_yolo26n-seg_int8.tflite 3.15 MB
outdoor_surface_guide_yolo26n-seg_integer_quant.tflite 3.16 MB
